# o1 PRM-Search Notebook

author： [xiaodongguaAIGC](https://github.com/dhcode-cpp)

1. 实现step-wise数据处理
2. 实现基于step-wise的SFT训练
4. 初始化PRM Model，实现PRM训练数据处理
5. 实现PRM Verfiy Sequence
6. 实现PRM-Search， 交替Step-Wise SFT生成和PRM检验。

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
torch.manual_seed(42)

# dataset

In [2]:
datasets = load_dataset('qgallouedec/prm800k')

Using the latest cached version of the dataset since qgallouedec/prm800k couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/denghang/.cache/huggingface/datasets/qgallouedec___prm800k/default/0.0.0/980ffdc8e4a05211ce8115215c1605d8c1428664 (last modified on Fri Dec 27 14:16:12 2024).


In [3]:
print(datasets)
print(datasets['train'][0])

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion', 'labels'],
        num_rows: 37482
    })
    test: Dataset({
        features: ['prompt', 'completion', 'labels'],
        num_rows: 3695
    })
})

{
    'prompt': [{'content': 'How many seconds are in 7.8 minutes?', 'role': 'user'}],
    'completion': [
        {'content': '7.8 minutes is the same as 7 minutes and 0.8 minutes.', 'role': 'assistant'},
        {
            'content': 'Right, and since there are 60 seconds in a minute, then there are 60 * 7 = 420 seconds in 7
minutes.',
            'role': 'assistant'
        },
        {
            'content': 'And since there are 60 seconds in a minute, then there are 60 * 0.8 = 48 seconds in 0.8 
minutes.',
            'role': 'assistant'
        },
        {'content': 'So, in total, there are 420 + 48 = 468 seconds in 7.8 minutes.', 'role': 'assistant'},
        {
            'content': "Right. Let's check our work. 7.8 minutes is the same as 7 minutes and 0.8 minutes.",
            'role': 'assistant'
        }
    ],
    'labels': [True, True, True, True, False]
}

In [4]:
# 数据太多重复
def show_item(idx, datasets):
    print('-'*100)
    print(datasets['train'][idx]['prompt'][0]['content'])
    for step in datasets['train'][idx]['completion']:
        print('****')
        print(step['content'])
    print(datasets['train'][idx]['labels'])


show_item(0, datasets)
show_item(1, datasets)
show_item(2, datasets)
show_item(10, datasets)
show_item(100, datasets)

----------------------------------------------------------------------------------------------------

How many seconds are in 7.8 minutes?

****

7.8 minutes is the same as 7 minutes and 0.8 minutes.

****

...

****

That's true. So, let's first find the largest three-digit integer with the desired property.

****

The first digit can't be 0, but it can be 1.

[True, True, True, True, True, True, True, False]

# Model

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = 'xiaodongguaAIGC/llama-3-debug'
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(
    model_name, add_bos_token=False, padding=True, truncation=True)
# print(model)
print(tokenizer.all_special_tokens)
# print(tokenizer)

['<|begin_of_text|>', '<|end_of_text|>']

In [6]:
print(tokenizer.all_special_tokens)
print(tokenizer.all_special_ids)
print(tokenizer.all_special_tokens_extended)

['<|begin_of_text|>', '<|end_of_text|>']

[128000, 128001]

[
    AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
    AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True)
]

In [7]:
# tokenizer.
tokenizer.add_special_tokens({
    'pad_token': '<|reserved_special_token_0|>',
    'sep_token': '<|reserved_special_token_1|>',
})


print(tokenizer.all_special_tokens)
print(tokenizer.all_special_ids)
print(tokenizer.all_special_tokens_extended)
# print(tokenizer)

SEP_TOKEN = '<|reserved_special_token_1|>'
PAD_TOKEN = '<|reserved_special_token_0|>'

SEP_TOKEN_ID = 128003
print(SEP_TOKEN_ID)

['<|begin_of_text|>', '<|end_of_text|>', '<|reserved_special_token_1|>', '<|reserved_special_token_0|>']

[128000, 128001, 128003, 128002]

[
    AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
    AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
    AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, 
special=True),
    AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, 
special=True)
]

128003

# Step-Wise SFT 

## format

```
# SYSTEM: you should step-wise answer math question. step with special token
# USER:{Question}
# ASSISTANT:{step1}{SEP_TOKEN},{step2}{SEP_TOKEN},...,{stepN}{SEP_TOKEN}{EOS_TOKEN}
```

In [8]:
tokenizer('hello world', return_tensors='pt')['input_ids'].shape
# tokenizer.encode('hello world', add_bos_token = False)
a = tokenizer.encode('hello world', return_tensors='pt')
# torch.cat((a,a), dim=1)
a

tensor([[128000,  15339,   1917]])

In [9]:

system_prompt = '#SYSTEM:you should step-wise answer math question. step with special token\n'


def format_template(system_prompt, question):
    return system_prompt + '#USER:' + question + '\n' + '#ASSISTANT:'

# not batched


def format_step_sft(example):
    question = example['prompt']
    steps = example['completion']
    # labels = example['labels']

    # 分词prompt
    # prompt_format = system_prompt + '#USER:' + question + '\n' + '#ASSISTANT:'
    prompt_format = format_template(system_prompt, question)
    prompt_encode = tokenizer.encode(prompt_format, return_tensors='pt')[0]
    prompt_len = prompt_encode.shape[0]

    # 分词response
    response = ''
    for step in steps:
        response = response + step + SEP_TOKEN
    response = response + tokenizer.eos_token
    response_encode = tokenizer.encode(response, return_tensors='pt')[0]
    response_encode = response_encode[1:]

    # 拼接 prompt + response
    input_ids = torch.cat((prompt_encode, response_encode), dim=0)
    attention_mask = torch.ones_like(input_ids)

    # 创建 label
    prompt_label = torch.clone(input_ids)
    prompt_label[:prompt_len] = -100  # 非response ignore
    labels = torch.roll(prompt_label, shifts=-1)  # label左移动一位，形成casual mask

    example['input_ids'] = input_ids
    example['attention_mask'] = attention_mask
    example['label_ids'] = labels
    # example['len']=prompt_len

    return example


print(format_step_sft(datasets['train'][0]))

{
    'prompt': [{'content': 'How many seconds are in 7.8 minutes?', 'role': 'user'}],
    'completion': [
        {'content': '7.8 minutes is the same as 7 minutes and 0.8 minutes.', 'role': 'assistant'},
        {
            'content': 'Right, and since there are 60 seconds in a minute, then there are 60 * 7 = 420 seconds in 7
minutes.',
            'role': 'assistant'
        },
        {
            'content': 'And since there are 60 seconds in a minute, then there are 60 * 0.8 = 48 seconds in 0.8 
minutes.',
            'role': 'assistant'
        },
        {'content': 'So, in total, there are 420 + 48 = 468 seconds in 7.8 minutes.', 'role': 'assistant'},
        {
            'content': "Right. Let's check our work. 7.8 minutes is the same as 7 minutes and 0.8 minutes.",
            'role': 'assistant'
        }
    ],
    'labels': [True, True, True, True, False],
    'input_ids': tensor([128000,      2,  47587,     25,   9514,   1288,   3094,  45539,   4320,
          7033,   3488,     13,   3094,    449,   3361,   4037,    198,      2,
          6584,     25,   4438,   1690,   6622,    527,    304,    220,     22,
            13,     23,   4520,   5380,      2,   5045,   3931,   2891,     25,
            22,     13,     23,   4520,    374,    279,   1890,    439,    220,
            22,   4520,    323,    220,     15,     13,     23,   4520,     13,
        128003,   6107,     11,    323,   2533,   1070,    527,    220,   1399,
          6622,    304,    264,   9568,     11,   1243,   1070,    527,    220,
          1399,    353,    220,     22,    284,    220,  12819,   6622,    304,
           220,     22,   4520,     13, 128003,   3112,   2533,   1070,    527,
           220,   1399,   6622,    304,    264,   9568,     11,   1243,   1070,
           527,    220,   1399,    353,    220,     15,     13,     23,    284,
           220,   2166,   6622,    304,    220,     15,     13,     23,   4520,
            13, 128003,   4516,     11,    304,   2860,     11,   1070,    527,
           220,  12819,    489,    220,   2166,    284,    220,  20304,   6622,
           304,    220,     22,     13,     23,   4520,     13, 128003,   6107,
            13,   6914,    596,   1817,   1057,    990,     13,    220,     22,
            13,     23,   4520,    374,    279,   1890,    439,    220,     22,
          4520,    323,    220,     15,     13,     23,   4520,     13, 128003,
        128001]),
    'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1]),
    'label_ids': tensor([  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,     22,
            13,     23,   4520,    374,    279,   1890,    439,    220,     22,
          4520,    323,    220,     15,     13,     23,   4520,     13, 128003,
          6107,     11,    323,   2533,   1070,    527,    220,   1399,   6622,
           304,    264,   9568,     11,   1243,   1070,    527,    220,   1399,
           353,    220,     22,    284,    220,  12819,   6622,    304,    220,
            22,   4520,     13, 128003,   3112,   2533,   1070,    527,    220,
          1399,   6622,    304,    264,   9568,     11,   1243,   1070,    527,
           220,   1399,    353,    220,     15,     13,   

In [10]:
datasets_format = datasets.map(format_step_sft,
                               # batched=True,
                               num_proc=24,
                               remove_columns=[
                                   'prompt', 'completion', 'labels']
                               )

In [11]:
# print(datasets_format['train'][0])

In [12]:
from transformers import DataCollatorWithPadding, PreTrainedTokenizerBase, DataCollatorForTokenClassification
data_collator = DataCollatorWithPadding(tokenizer=tokenizer,

                                        )
# batch = data_collator(datasets_format['test'])

In [13]:
import transformers
from typing import Dict, List, Optional, Union, Any


class DataCollatorForSFT(DataCollatorWithPadding):
    """
    继承DataCollatorWithPadding实现动态padding
    """
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str,
                   transformers.tokenization_utils_base.PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    return_tensors: str = "pt"

    # features: List[Dict[str, Union[List[int], torch.Tensor]]]

    # def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 分离input和label
        input_ids = [{"input_ids": f["input_ids"]} for f in features]

        # 动态padding input
        batch = self.tokenizer.pad(
            input_ids,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors=self.return_tensors,
        )

        labels_list = [f["label_ids"] for f in features]
        max_len = max([len(labels) for labels in labels_list])
        # print(max_len)
        target_labels = []

        for labels in labels_list:
            labels = labels + [-100] * (max_len - len(labels))
            target_labels.append(labels)
        batch["labels"] = torch.tensor(target_labels)

        batch['input_ids'].to('cpu')
        batch['labels'].to('cpu')
        batch['attention_mask'].to('cpu')

        return batch


data_collator = DataCollatorForSFT(tokenizer=tokenizer,
                                   padding='longest',
                                   max_length=4096
                                   )
batch = data_collator(datasets_format['test'])
print(batch)

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{
    'input_ids': tensor([[128000,      2,  47587,  ..., 128002, 128002, 128002],
        [128000,      2,  47587,  ..., 128002, 128002, 128002],
        [128000,      2,  47587,  ..., 128002, 128002, 128002],
        ...,
        [128000,      2,  47587,  ..., 128002, 128002, 128002],
        [128000,      2,  47587,  ..., 128002, 128002, 128002],
        [128000,      2,  47587,  ..., 128002, 128002, 128002]]),
    'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]),
    'labels': tensor([[-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100],
        ...,
        [-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100]])
}

In [14]:
from torch.utils.data import DataLoader, Dataset

data_loader = DataLoader(
    datasets_format['test'],                    # 数据集
    batch_size=2,             # 批次大小
    shuffle=True,              # 是否打乱数据
    collate_fn=data_collator,           # 自定义数据整理函数
)
model.to('cpu')
for batch in data_loader:
    # print(batch)
    output = model.forward(**batch)
    break

print(output)

CausalLMOutputWithPast(
    loss=tensor(11.7752, grad_fn=<NllLossBackward0>),
    logits=tensor([[[ 0.1592, -0.0005, -0.0272,  ...,  0.1128,  0.1196, -0.0503],
         [ 0.0244,  0.1523,  0.1641,  ..., -0.0840, -0.0786,  0.1196],
         [ 0.1465, -0.0198,  0.1108,  ..., -0.1021,  0.1895,  0.1943],
         ...,
         [ 0.0815,  0.2559,  0.1147,  ..., -0.2578, -0.1855,  0.0591],
         [ 0.1885,  0.3281, -0.0259,  ..., -0.2393,  0.0859, -0.0522],
         [ 0.0088,  0.1357, -0.0574,  ..., -0.0097, -0.2197,  0.1260]],

        [[ 0.1592, -0.0005, -0.0272,  ...,  0.1128,  0.1196, -0.0503],
         [ 0.0244,  0.1523,  0.1641,  ..., -0.0840, -0.0786,  0.1196],
         [ 0.1465, -0.0198,  0.1108,  ..., -0.1021,  0.1895,  0.1943],
         ...,
         [ 0.1367, -0.2217, -0.0295,  ..., -0.0679,  0.0131,  0.0146],
         [ 0.1357, -0.2217, -0.0294,  ..., -0.0684,  0.0135,  0.0142],
         [ 0.1357, -0.2227, -0.0294,  ..., -0.0684,  0.0135,  0.0145]]],
       dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>),
    past_key_values=(
        (
            tensor([[[[ 0.0654,  0.0605, -0.1167,  ..., -0.0483,  0.2969, -0.0742],
          [ 0.0349, -0.0525,  0.0757,  ..., -0.0952,  0.2930, -0.1514],
          [ 0.3242, -0.2656,  0.0262,  ..., -0.0417,  0.0732, -0.1865],
          ...,
          [ 0.0776,  0.1406, -0.0942,  ..., -0.1045, -0.1904, -0.0815],
          [ 0.1113, -0.1309,  0.1602,  ...,  0.0811,  0.7461, -0.0972],
          [ 0.1226,  0.2852, -0.0796,  ...,  0.0571,  0.1553,  0.0027]],

         [[-0.1416, -0.0056, -0.0415,  ...,  0.1689, -0.0266, -0.1582],
          [-0.2559, -0.0049,  0.1816,  ...,  0.0625,  0.0525,  0.0068],
          [-0.1387, -0.0532, -0.1680,  ..., -0.1543,  0.1719,  0.2793],
          ...,
          [ 0.0435, -0.0155, -0.3457,  ...,  0.1455,  0.0056, -0.0171],
          [ 0.2314, -0.0723,  0.1680,  ..., -0.0603, -0.0017, -0.0535],
          [ 0.0571,  0.1963, -0.3867,  ...,  0.0009, -0.0991, -0.2910]]],


        [[[ 0.0654,  0.0605, -0.1167,  ..., -0.0483,  0.2969, -0.0742],
          [ 0.0349, -0.0525,  0.0757,  ..., -0.0952,  0.2930, -0.1514],
          [ 0.3242, -0.2656,  0.0262,  ..., -0.0417,  0.0732, -0.1865],
          ...,
          [-0.0015, -0.0483,  0.2227,  ...,  0.2793,  0.3262, -0.1582],
          [ 0.1602, -0.0967,  0.1943,  ...,  0.2793,  0.3262, -0.1582],
          [ 0.1748, -0.1260,  0.1602,  ...,  0.2793,  0.3262, -0.1582]],

         [[-0.1416, -0.0056, -0.0415,  ...,  0.1689, -0.0266, -0.1582],
          [-0.2559, -0.0049,  0.1816,  ...,  0.0625,  0.0525,  0.0068],
          [-0.1387, -0.0532, -0.1680,  ..., -0.1543,  0.1719,  0.2793],
          ...,
          [-0.0415,  0.1768,  0.2637,  ...,  0.3672,  0.3516, -0.3652],
          [-0.2129,  0.1328,  0.1973,  ...,  0.3672,  0.3516, -0.3652],
          [-0.1875,  0.0645,  0.1250,  ...,  0.3672,  0.3516, -0.3652]]]],
       dtype=torch.bfloat16, grad_fn=<AddBackward0>),
            tensor([[[[ 0.1582, -0.1172, -0.1357,  ...,  0.1113, -0.1299, -0.2275],
          [-0.1011,  0.3203, -0.2812,  ..., -0.1514, -0.2188, -0.0221],
          [-0.0437, -0.1060, -0.1709,  ...,  0.0708,  0.0006,  0.1201],
          ...,
          [-0.0559,  0.0986,  0.2676,  ..., -0.0310, -0.0564, -0.0090],
          [-0.1455, -0.0598, -0.1553,  ..., -0.0277, -0.2695,  0.0352],
          [ 0.1934, -0.2227, -0.1416,  ..., -0.0430,  0.1045,  0.0356]],

         [[ 0.0243, -0.0042, -0.1030,  ..., -0.0322,  0.1396,  0.0762],
          [ 0.0718,  0.2217,  0.1797,  ...,  0.1055, -0.0496, -0.2021],
          [-0.3652,  0.1069, -0.0430,  ..., -0.3164, -0.0525, -0.2715],
          ...,
          [-0.0208, -0.1572, -0.0491,  ...,  0.2871,  0.1738, -0.0035],
          [ 0.1924, -0.2578,  0.0005,  ...,  0.1592,  0.0203,  0.0771],
          [-0.1455,  0.0361,  0.1748,  ...,  0.0845, -0.0527, -0.2695]]],


        [[[ 0.1582, -0.1172, -0.1357,  ...,  0.1113, -0.1299, -0.2275],
          [-0.1011,  0.3203, -0.2812,  ..., -0.1514, -0.2188, -0.0221],
          [-0

## training

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="your-model",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    no_cuda=True,
    remove_unused_columns=False,
)

/Users/denghang/Miniconda3/envs/llm/lib/python3.11/site-packages/transformers/training_args.py:1590: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions, references=labels)

In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets_format["train"],
    data_collator=data_collator,
    # compute_metrics =compute_metrics,
)

In [18]:
# trainer.train()

In [19]:
output_sft_path = './../output/Llama-3-step-sft'

In [20]:
trainer.save_model(output_sft_path)
tokenizer.save_pretrained(output_sft_path)

('./../output/Llama-3-step-sft/tokenizer_config.json',
 './../output/Llama-3-step-sft/special_tokens_map.json',
 './../output/Llama-3-step-sft/tokenizer.json')

## inference

In [21]:
question = "how to solve y=x^2+2x+1"
prompt = format_template(system_prompt, question)
input = tokenizer.encode(prompt, return_tensors='pt')

In [22]:
print(input)

tensor([[128000,      2,  47587,     25,   9514,   1288,   3094,  45539,   4320,
           7033,   3488,     13,   3094,    449,   3361,   4037,    198,      2,
           6584,     25,   5269,    311,  11886,    379,  26459,     61,     17,
             10,     17,     87,     10,     16,    198,      2,   5045,   3931,
           2891,     25]])

In [23]:
def generate_greed(input, max_tokens=10,
                   temperature=0.9,
                   past_key_values=None):
    i = 0
    result = []
    model.eval()
    for i in range(max_tokens):
        # print(input)
        with torch.no_grad():
            # greedy search with KV Cache
            output = model(input,
                           past_key_values=past_key_values,
                           use_cache=True
                           )

            logits = output.logits[0, -1, :] / temperature  # do sample
            past_key_values = output.past_key_values

        probs = torch.softmax(logits, dim=-1)
        next_token_idx = torch.multinomial(probs, num_samples=1)  # do sample
        # print(next_token_idx)
        result.append(next_token_idx.item())
        input = next_token_idx.unsqueeze(dim=0)

        if next_token_idx == tokenizer.eos_token_id:
            break

    return result


result = generate_greed(input)

print(result)

result_string = tokenizer.decode(result)

print(result_string)

We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)


[49458, 114999, 14163, 41894, 32958, 73058, 12761, 30333, 119741, 73251]

retrosişilymp.track_extra depreci SolutionırĠ anonymously

# Step-Wise PRM

follow "Let's verify step-by-step"

## PRM Model

In [24]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = output_sft_path

# PRM Model复用语言模型，而不用专门去改变分类头
prm_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16)
prm_tokenizer = AutoTokenizer.from_pretrained(
    model_name, add_bos_token=False, padding=True, truncation=True)

In [25]:
id = prm_tokenizer('Positive', add_special_tokens=False)
print(id)
print(prm_tokenizer.decode([36590]))

{'input_ids': [36590], 'attention_mask': [1]}

Positive

In [26]:
# PRM Model的分类头有128k类别，那么我们只要关注以下token对应的分类类别
# 这样我们就可以复用原分类头参数，且复用“SFT”数据和训练目标了

positive_id = prm_tokenizer('Positive', add_special_tokens=False)['input_ids']
positive_token = 'Positive'
print(positive_id)

negative_id = prm_tokenizer('Negative', add_special_tokens=False)['input_ids']
negative_token = 'Negative'
print(negative_id)

neutral_id = prm_tokenizer('Neutral', add_special_tokens=False)['input_ids']
neutral_token = 'Neutral'
print(neutral_id)

[36590]

[39589]

[88007]

## Format

In [27]:
# 针对打分，重写prompt
prm_system_prompt = '#SYSTEM:you should step-wise score the correctness of answer step \n'


def format_prm_template(system_prompt, question):
    return system_prompt + '#USER:' + question + '\n' + '#ASSISTANT:'

In [28]:
# not batched

label_map = {0: negative_id, 1: positive_id}
# label_token_map = [0: Negative, 1: Positive]


def format_step_prm(example):
    question = example['prompt']
    steps = example['completion']
    labels = example['labels']

    # 分词prompt
    # prompt_format = system_prompt + '#USER:' + question + '\n' + '#ASSISTANT:'
    prompt_format = format_template(prm_system_prompt, question)
    prompt_encode = tokenizer.encode(prompt_format, return_tensors='pt')[0]
    prompt_len = prompt_encode.shape[0]

    response_token_ids = []
    place_indexs = []
    label_idx = []

    for step, label in zip(steps, labels):
        response = step + SEP_TOKEN  # step的尾部加上sep token
        step_response_token_ids = tokenizer.encode(
            response, add_special_tokens=False)

        response_token_ids.extend(step_response_token_ids)
        place_indexs.append(len(response_token_ids) + prompt_len)
        label_idx.append(label_map[label][0])  # bug?

    response_token_ids.extend([tokenizer.eos_token_id])  # 完整结束后增加eos token
    response_token_ids = torch.tensor(response_token_ids)

    # 拼接 prompt + response
    input_ids = torch.cat((prompt_encode, response_token_ids), dim=0)
    attention_mask = torch.ones_like(input_ids)
    # attention_mask[place_indexs] = False

    # 创建 label， 在sep token里才有回归的标签
    # 这里相当于一长串数据，可以一次性回归多个sep_token 对应的correctness的标签
    print(label_idx)
    place_indexs = [idx - 1 for idx in place_indexs]
    prompt_label = torch.ones_like(input_ids) * -100
    prompt_label[place_indexs] = torch.tensor(label_idx, dtype=torch.long)

    example['input_ids'] = input_ids
    example['attention_mask'] = attention_mask
    example['label_ids'] = prompt_label

    return example


print(format_step_prm(datasets['train'][0]))

[36590, 36590, 36590, 36590, 39589]

{
    'prompt': [{'content': 'How many seconds are in 7.8 minutes?', 'role': 'user'}],
    'completion': [
        {'content': '7.8 minutes is the same as 7 minutes and 0.8 minutes.', 'role': 'assistant'},
        {
            'content': 'Right, and since there are 60 seconds in a minute, then there are 60 * 7 = 420 seconds in 7
minutes.',
            'role': 'assistant'
        },
        {
            'content': 'And since there are 60 seconds in a minute, then there are 60 * 0.8 = 48 seconds in 0.8 
minutes.',
            'role': 'assistant'
        },
        {'content': 'So, in total, there are 420 + 48 = 468 seconds in 7.8 minutes.', 'role': 'assistant'},
        {
            'content': "Right. Let's check our work. 7.8 minutes is the same as 7 minutes and 0.8 minutes.",
            'role': 'assistant'
        }
    ],
    'labels': [True, True, True, True, False],
    'input_ids': tensor([128000,      2,  47587,     25,   9514,   1288,   3094,  45539,   5573,
           279,  58423,    315,   4320,   3094,    720,      2,   6584,     25,
          4438,   1690,   6622,    527,    304,    220,     22,     13,     23,
          4520,   5380,      2,   5045,   3931,   2891,     25,     22,     13,
            23,   4520,    374,    279,   1890,    439,    220,     22,   4520,
           323,    220,     15,     13,     23,   4520,     13, 128003,   6107,
            11,    323,   2533,   1070,    527,    220,   1399,   6622,    304,
           264,   9568,     11,   1243,   1070,    527,    220,   1399,    353,
           220,     22,    284,    220,  12819,   6622,    304,    220,     22,
          4520,     13, 128003,   3112,   2533,   1070,    527,    220,   1399,
          6622,    304,    264,   9568,     11,   1243,   1070,    527,    220,
          1399,    353,    220,     15,     13,     23,    284,    220,   2166,
          6622,    304,    220,     15,     13,     23,   4520,     13, 128003,
          4516,     11,    304,   2860,     11,   1070,    527,    220,  12819,
           489,    220,   2166,    284,    220,  20304,   6622,    304,    220,
            22,     13,     23,   4520,     13, 128003,   6107,     13,   6914,
           596,   1817,   1057,    990,     13,    220,     22,     13,     23,
          4520,    374,    279,   1890,    439,    220,     22,   4520,    323,
           220,     15,     13,     23,   4520,     13, 128003, 128001]),
    'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1]),
    'label_ids': tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100, 36590,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100, 36590,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100, 36590,  -100,  -100,  -100,
         -100,  -100,  -100,  -1

In [29]:
print(prm_tokenizer.decode([36590]))
print(prm_tokenizer.decode([39589]))

Positive

Negative

## prm dataset

In [30]:
datasets_format_prm = datasets.map(format_step_prm,
                                   # batched=True,
                                   num_proc=24,
                                   remove_columns=[
                                       'prompt', 'completion', 'labels']
                                   )

In [31]:
# 复用SFT数据和收集器
data_collator = DataCollatorForSFT(tokenizer=prm_tokenizer,
                                   padding='longest',
                                   max_length=4096
                                   )
batch = data_collator(datasets_format['test'])  # 小数据集用于调试

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [32]:
# print(batch.input_ids[0,:].tolist())
# print(batch.attention_mask[0,:])
# print(batch.labels[0,:])

## Training

复用sft的训练trainer

In [33]:
from transformers import Trainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="your-model",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    no_cuda=True,
    remove_unused_columns=False,
)


trainer = Trainer(
    model=prm_model,
    args=training_args,
    train_dataset=datasets_format_prm["train"],
    data_collator=data_collator,
    # compute_metrics =compute_metrics,
)

/Users/denghang/Miniconda3/envs/llm/lib/python3.11/site-packages/transformers/training_args.py:1590: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


In [34]:
# trainer.train()

In [35]:
output_prm_path = './../output/Llama-3-step-prm'

In [36]:
trainer.save_model(output_prm_path)
prm_tokenizer.save_pretrained(output_prm_path)

('./../output/Llama-3-step-prm/tokenizer_config.json',
 './../output/Llama-3-step-prm/special_tokens_map.json',
 './../output/Llama-3-step-prm/tokenizer.json')

## Inference

我们如果先用SFT模型产生了step-wise的解答，

我们并不需要一步步的用prm来verify，

比如sft生成了5个步骤，那么PRM可以并行给5个sep token对应的预测进行分类

In [37]:
# 我们先拿数据集已有的数据来做预测

label_map = {0: negative_id, 1: positive_id}
# label_token_map = [0: Negative, 1: Positive]


def format_step_prm(example):
    question = example['prompt'][0]['content']
    steps = example['completion']
    labels = example['labels']

    # 分词prompt
    # prompt_format = system_prompt + '#USER:' + question + '\n' + '#ASSISTANT:'
    prompt_format = format_template(system_prompt, question)
    prompt_encode = tokenizer.encode(prompt_format, return_tensors='pt')[0]
    prompt_len = prompt_encode.shape[0]

    response_token_ids = []
    place_indexs = []
    label_idx = []

    for step, label in zip(steps, labels):
        response = step['content'] + SEP_TOKEN
        step_response_token_ids = tokenizer.encode(
            response, add_special_tokens=False)
        response_token_ids.extend(step_response_token_ids)
        place_indexs.append(len(response_token_ids) + prompt_len)
        label_idx.append(label_map[label][0])

    response_token_ids.extend([tokenizer.eos_token_id])
    response_token_ids = torch.tensor(response_token_ids)

    # 拼接 prompt + response
    input_ids = torch.cat((prompt_encode, response_token_ids), dim=0)
    attention_mask = torch.ones_like(input_ids)

    # 创建 label
    print(label_idx)
    place_indexs = [idx - 1 for idx in place_indexs]
    prompt_label = torch.ones_like(input_ids) * -100
    prompt_label[place_indexs] = torch.tensor(label_idx)

    example['input_ids'] = input_ids
    example['attention_mask'] = attention_mask
    example['label_ids'] = prompt_label

    return example


print(format_step_prm(datasets['train'][0]))

[36590, 36590, 36590, 36590, 39589]

{
    'prompt': [{'content': 'How many seconds are in 7.8 minutes?', 'role': 'user'}],
    'completion': [
        {'content': '7.8 minutes is the same as 7 minutes and 0.8 minutes.', 'role': 'assistant'},
        {
            'content': 'Right, and since there are 60 seconds in a minute, then there are 60 * 7 = 420 seconds in 7
minutes.',
            'role': 'assistant'
        },
        {
            'content': 'And since there are 60 seconds in a minute, then there are 60 * 0.8 = 48 seconds in 0.8 
minutes.',
            'role': 'assistant'
        },
        {'content': 'So, in total, there are 420 + 48 = 468 seconds in 7.8 minutes.', 'role': 'assistant'},
        {
            'content': "Right. Let's check our work. 7.8 minutes is the same as 7 minutes and 0.8 minutes.",
            'role': 'assistant'
        }
    ],
    'labels': [True, True, True, True, False],
    'input_ids': tensor([128000,      2,  47587,     25,   9514,   1288,   3094,  45539,   4320,
          7033,   3488,     13,   3094,    449,   3361,   4037,    198,      2,
          6584,     25,   4438,   1690,   6622,    527,    304,    220,     22,
            13,     23,   4520,   5380,      2,   5045,   3931,   2891,     25,
            22,     13,     23,   4520,    374,    279,   1890,    439,    220,
            22,   4520,    323,    220,     15,     13,     23,   4520,     13,
        128003,   6107,     11,    323,   2533,   1070,    527,    220,   1399,
          6622,    304,    264,   9568,     11,   1243,   1070,    527,    220,
          1399,    353,    220,     22,    284,    220,  12819,   6622,    304,
           220,     22,   4520,     13, 128003,   3112,   2533,   1070,    527,
           220,   1399,   6622,    304,    264,   9568,     11,   1243,   1070,
           527,    220,   1399,    353,    220,     15,     13,     23,    284,
           220,   2166,   6622,    304,    220,     15,     13,     23,   4520,
            13, 128003,   4516,     11,    304,   2860,     11,   1070,    527,
           220,  12819,    489,    220,   2166,    284,    220,  20304,   6622,
           304,    220,     22,     13,     23,   4520,     13, 128003,   6107,
            13,   6914,    596,   1817,   1057,    990,     13,    220,     22,
            13,     23,   4520,    374,    279,   1890,    439,    220,     22,
          4520,    323,    220,     15,     13,     23,   4520,     13, 128003,
        128001]),
    'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1]),
    'label_ids': tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100, 36590,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100, 36590,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100, 36590,  -100,
  

In [38]:
test_data = format_step_prm(datasets['test'][0])
# print(test_data)
idx = torch.where(test_data['label_ids'] != -100)
# print(idx)
# print(test_data['input_ids'][idx]) # sep token

[36590, 39589]

In [39]:
# 找到sep token对应的索引，即是我们的预测目标
idx = torch.where(test_data['input_ids'] == 128003)[0]
print(idx)

tensor([ 98, 126])

In [40]:
# 验证时做一次forward
prm_model.eval()
with torch.no_grad():
    logits = prm_model(input_ids=test_data['input_ids'].unsqueeze(dim=0),
                       attention_mask=test_data['attention_mask'].unsqueeze(dim=0), ).logits

In [41]:
# 取出sep token的losits
print(logits.shape)
sep_logits = logits[0, idx, :]  # sep_token对应的logits的预测
print(sep_logits.shape)
print(sep_logits[0, positive_id])
print(sep_logits[0, negative_id])

torch.Size([1, 128, 128256])

torch.Size([2, 128256])

tensor([-0.0223], dtype=torch.bfloat16)

tensor([-0.1572], dtype=torch.bfloat16)

In [42]:
# 判别分类概率
sep_logits_class = sep_logits[:, [positive_id, negative_id]].squeeze(
    dim=2)  # 只取positive，negative类别对应的概率
print(sep_logits_class.shape)
print(sep_logits_class)

sep_prob = torch.nn.functional.softmax(sep_logits_class, dim=1)
print(sep_prob)
_, pred = torch.max(sep_prob, dim=1)

print(test_data['labels'])  # labels
print(pred)  # 0: negative, 1: positive

torch.Size([2, 2])

tensor([[-0.0223, -0.1572],
        [-0.0216, -0.1592]], dtype=torch.bfloat16)

tensor([[0.5352, 0.4668],
        [0.5352, 0.4648]], dtype=torch.bfloat16)

[True, False]

tensor([0, 0])

# PRM-Search

## Greedy Search

1. 我们在4.5中，对于一个问题，先generate到EOS，再做PRM判别
2. 现在我们按照step-wise的交替“生成步骤”和“验证”，直至到达eos，或者到达最大步数

TODO：如何借助KV-Cache(past-key-value), 来避免重复计算。

In [43]:
question = 'how to solve x^2+2x+1 = 0'

# SFT和PRM的输入有不同的模版，即有不同的输入

prompt_format = format_template(system_prompt, question)
input = tokenizer.encode(prompt_format, return_tensors='pt')

prompt_format_prm = format_template(prm_system_prompt, question)
input_prm = tokenizer.encode(prompt_format_prm, return_tensors='pt')

In [44]:
print(input_prm)

tensor([[128000,      2,  47587,     25,   9514,   1288,   3094,  45539,   5573,
            279,  58423,    315,   4320,   3094,    720,      2,   6584,     25,
           5269,    311,  11886,    865,     61,     17,     10,     17,     87,
             10,     16,    284,    220,     15,    198,      2,   5045,   3931,
           2891,     25]])

In [80]:
# 验证函数，我们默认输入数据格式为：xxxxxxxxxxxx<SEP>, 即token idx序列的最后一个token是SEP token
# prm_past_key_values为KV Cache，减少推理耗时。
def verify_function(input_prm=input, prm_model=model, prm_past_key_values=None):
    prm_model.eval()
    with torch.no_grad():
        # print(input_prm)
        output = prm_model(input_ids=input_prm,
                           past_key_values=prm_past_key_values,
                           use_cache=True,  # TODO: use kv-cache avoid recompute
                           )
        last_logits = output.logits[0, -1, :]
        past_key_values = output.past_key_values  # new kv cache

    # 两个类别来做softmax，将logits 转为 prob
    p_positive = torch.exp(last_logits[positive_id]) / (
        torch.exp(last_logits[positive_id]) + torch.exp(last_logits[negative_id]))
    p_negative = 1 - p_positive

    if p_positive > p_negative:
        return True, p_positive.item(), p_negative.item(), prm_past_key_values
    else:
        return False, p_positive.item(), p_negative.item(), prm_past_key_values


result, p_pos, p_neg, _ = verify_function(input_prm, prm_model)
print(result)
print(p_pos)
print(p_neg)

True

0.578125

0.421875

In [81]:
import copy

In [82]:
# past_key_values为KV Cache，减少推理耗时。
def generate_greedy_step(input=input,
                         model=None,
                         max_tokens=10,
                         temperature=0.9,
                         past_key_values=None):
    i = 0
    result = []
    model.eval()
    new_past_key_values = copy.deepcopy(past_key_values)
    for i in range(max_tokens):
        model.eval()
        with torch.no_grad():
            # greedy search with KV Cache
            output = model(input_ids=input,
                           past_key_values=new_past_key_values,
                           use_cache=False  # TODO: use kv-cache
                           )

            logits = output.logits[0, -1, :] / temperature  # do sample
            new_past_key_values = output.past_key_values
            # print(output.past_key_values[0][0].shape)

        probs = torch.softmax(logits, dim=-1)
        next_token_idx = torch.multinomial(probs, num_samples=1)  # do sample
        # print(next_token_idx)
        result.append(next_token_idx.item())
        input = next_token_idx.unsqueeze(dim=0)

        if next_token_idx == SEP_TOKEN_ID:
            return result, new_past_key_values

    # 如果到达生成长度也没有sep token，我们“强制”手动加， 实际上模型可以推断出sep token出来
    result.append(SEP_TOKEN_ID)

    return result, new_past_key_values


result, _ = generate_greedy_step(input, model)

print(result)

result_string = tokenizer.decode(result)

print(result_string)

[112833, 82203, 86206, 4126, 23159, 7897, 67992, 67336, 18622, 82759, 128003]

леж toDate需要Edit_protaked IPO licens rewritetoPromise<|reserved_special_token_1|>

In [83]:
# 进入到PRM 搜索函数
# 1. 生成步骤
# 2. 检验步骤
# 3. 如果检验通过，那么就跳到1，并更新id序列（接收）+ KV Cache更新，否则就回退
# 增加生成step次数判断，避免生成太多次
def prm_search(input,
               input_prm,
               model,
               prm_model,
               max_step=10,
               accumulate_max_step=30,
               temperature=0.9,
               ):
    i = 0
    result = []
    model.eval()
    acc_max_step = 0
    past_key_values = None,
    prm_past_key_values = None,
    while i < max_step:

        print('i:', i)
        if input[0, -1] == tokenizer.eos_token_id:
            print('meet eos token, and finish prm search')
            break

        # 1. generate step
        step_idx, new_past_key_values = generate_greedy_step(input=input,
                                                             model=model,
                                                             past_key_values=None)

        step_idx_tensor = torch.tensor(
            step_idx, dtype=torch.long).unsqueeze(dim=0)

        # 2. verify step
        new_input_prm = torch.cat((input_prm, step_idx_tensor), dim=1)
        result, p_pos, p_neg, new_prm_past_key_values = verify_function(input_prm=new_input_prm,
                                                                        prm_model=prm_model,
                                                                        prm_past_key_values=None)
        print(result)
        print(p_pos, p_neg)

        # 3. update step
        # 当前步骤正确则更新输入序列和KV-Cache
        # 不正确不更新，而下一个loop的SFT生成由于是do_sample, 则可能采样出不一样的解答步骤
        if result:  # True update
            input = torch.cat((input_prm, step_idx_tensor), dim=1)
            input_prm = new_input_prm
            past_key_values = new_past_key_values
            prm_past_key_values = new_prm_past_key_values
            i = i + 1

        acc_max_step += 1
        if acc_max_step > accumulate_max_step:
            print('search many times to failed')
            return input

    return input


print(input)
result = prm_search(input, input_prm, model, prm_model)

tensor([[128000,      2,  47587,     25,   9514,   1288,   3094,  45539,   4320,
           7033,   3488,     13,   3094,    449,   3361,   4037,    198,      2,
           6584,     25,   5269,    311,  11886,    865,     61,     17,     10,
             17,     87,     10,     16,    284,    220,     15,    198,      2,
           5045,   3931,   2891,     25]])

i: 0

True

0.55078125 0.44921875

i: 1

...

True

0.5546875 0.4453125

i: 9

True

0.55078125 0.44921875

In [84]:
result_string = tokenizer.decode(result[0])
print(result_string)

<|begin_of_text|>#SYSTEM:you should step-wise score the correctness of answer step 
#USER:how to solve x^2+2x+1 = 0
#ASSISTANT:urbaley UV Beverage communicate execut curatedFIELD  source 
khiển<|reserved_special_token_1|>.junit悉walker засідKyle jdbc 
Verg_AXıyordu:message<|reserved_special_token_1|>odnímeasurement )}

 Dietadvanced �řez baby traveler functools<|reserved_special_token_1|>_vectorslicos단체 kraDue� переда speechesenus
Geek<|reserved_special_token_1|>еко lẽ輯 Calcul(flag Uruguayquisite堆时候 친<|reserved_special_token_1|>.wallet 
 broadcasterشي rise연_neighborsPosts eben значительно<|reserved_special_token_1|> Guantanamoernetes 
readinessstartTime MitBehaviour blend getApp Bou     dto<|reserved_special_token_1|> templ'r กล>d 
emoc383έντセンターComput utilizando<|reserved_special_token_1|> กลครอง مشاهده/api_NAMESPACE reliant&&&& 
charcoalforegroundColorpcl<|reserved_special_token_1|>不是TINGんですляли окруж Nikon ajout 내Telefone 
elder<|reserved_special_token_1|>

In [86]:
# 设定格式打印
result_line = result_string.replace(
    "<|reserved_special_token_1|>", " <SEP>\n\n")
print(result_line)

<|begin_of_text|>#SYSTEM:you should step-wise score the correctness of answer step 
#USER:how to solve x^2+2x+1 = 0
#ASSISTANT:urbaley UV Beverage communicate execut curatedFIELD  source khiển <SEP>

.junit悉walker засідKyle jdbc Verg_AXıyordu:message <SEP>

odnímeasurement )}

 Dietadvanced �řez baby traveler functools <SEP>

_vectorslicos단체 kraDue� переда speechesenus Geek <SEP>

еко lẽ輯 Calcul(flag Uruguayquisite堆时候 친 <SEP>

.wallet 
 broadcasterشي rise연_neighborsPosts eben значительно <SEP>

 Guantanamoernetes readinessstartTime MitBehaviour blend getApp Bou     dto <SEP>

 templ'r กล>d emoc383έντセンターComput utilizando <SEP>

 กลครอง مشاهده/api_NAMESPACE reliant&&&& charcoalforegroundColorpcl <SEP>

不是TINGんですляли окруж Nikon ajout 내Telefone elder <SEP>